# 领航者7020：真实 PYNQ Ascon 验证

从仓库根目录启动 Jupyter，并按 `Restart Kernel and Run All Cells` 顺序执行。
本 Notebook 只使用真实 Overlay 和 release 固件，没有模拟回退。
运行结果会写入 `data/pynq/notebook_<时间>/`，便于直接归档和报告引用。

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import pynq

repo = Path(os.environ.get("ASCON_REPO", "/home/xilinx/jupyter_notebooks/amd_fpga_2026")).resolve()
assert (repo / "src/pynq/board_demo.py").is_file(), "请设置 ASCON_REPO 指向实际仓库"
os.chdir(str(repo))
sys.path.insert(0, str(repo))

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = repo / "data/pynq" / ("notebook_" + run_id)
run_dir.mkdir(parents=True, exist_ok=False)
identity = {
    "generated_local": datetime.now().isoformat(timespec="seconds"),
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=str(repo), text=True
    ).strip(),
    "pynq_version": pynq.__version__,
    "python_version": platform.python_version(),
    "kernel": platform.platform(),
}
(run_dir / "identity.json").write_text(
    json.dumps(identity, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(identity, indent=2))

{
  "generated_local": "2026-10-31T11:20:52",
  "generated_utc": "2026-10-31T11:20:52+00:00",
  "git_commit": "ac9143d5c6acc9e7ac61f907a7e802eb6b35cddd",
  "pynq_version": "2.7.0",
  "python_version": "3.8.2",
  "kernel": "Linux-5.4.0-xilinx-v2020.2-armv7l-with-glibc2.29"
}


## 1. 实板容量内全量 KAT

当前 `RUN_FULL_KAT = True`，对 SW/HW/ISE 三模式分别运行容量内全部 289 组 KAT，
并逐组检查加密、解密和错误 Tag 拒绝。
只有实际通过时才写最终 JSON；失败会立即终止 Notebook。

In [2]:
import contextlib
import io
from src.pynq.board_demo import run

RUN_FULL_KAT = True
smoke_log = io.StringIO()
with contextlib.redirect_stdout(smoke_log), contextlib.redirect_stderr(smoke_log):
    smoke = run(
        "board/vivado_zynq7000_xc7z020_pynq/release",
        full_kat=RUN_FULL_KAT,
    )

print(smoke_log.getvalue())
assert smoke["board_validation"] == "PASS"
(run_dir / "pynq_board_smoke.json").write_text(
    json.dumps(smoke, indent=2, default=str) + "\n", encoding="utf-8"
)
(run_dir / "pynq_board_smoke.log").write_text(
    smoke_log.getvalue(), encoding="utf-8"
)
print("KAT evidence saved to", run_dir)

sw {'kat_pass': 289, 'tag_reject': 289, 'samples': [{'kat': 1, 'cycles': 17584}, {'kat': 2, 'cycles': 22867}, {'kat': 3, 'cycles': 22899}, {'kat': 4, 'cycles': 22911}, {'kat': 5, 'cycles': 22880}, {'kat': 6, 'cycles': 22907}, {'kat': 7, 'cycles': 22950}, {'kat': 8, 'cycles': 22964}, {'kat': 9, 'cycles': 22998}, {'kat': 10, 'cycles': 23090}, {'kat': 11, 'cycles': 23107}, {'kat': 12, 'cycles': 23120}, {'kat': 13, 'cycles': 23102}, {'kat': 14, 'cycles': 23116}, {'kat': 15, 'cycles': 23130}, {'kat': 16, 'cycles': 23141}, {'kat': 17, 'cycles': 27992}, {'kat': 34, 'cycles': 17973}, {'kat': 35, 'cycles': 23071}, {'kat': 36, 'cycles': 23101}, {'kat': 37, 'cycles': 23113}, {'kat': 38, 'cycles': 23080}, {'kat': 39, 'cycles': 23107}, {'kat': 40, 'cycles': 23150}, {'kat': 41, 'cycles': 23164}, {'kat': 42, 'cycles': 23196}, {'kat': 43, 'cycles': 23288}, {'kat': 44, 'cycles': 23307}, {'kat': 45, 'cycles': 23320}, {'kat': 46, 'cycles': 23304}, {'kat': 47, 'cycles': 23318}, {'kat': 48, 'cycles': 23332

## 2. 固定输入 warm benchmark

使用官方 KAT `#529`：PT=16 bytes、AD=0。每种模式先预热 5 次，再记录 30 次
加密和 30 次解密。每次调用都检查 KAT，`mcycle` scope 不包含 PS/UART 传输。

In [3]:
from src.pynq.board_benchmark import run_benchmark

benchmark_log = io.StringIO()
with contextlib.redirect_stdout(benchmark_log), contextlib.redirect_stderr(benchmark_log):
    benchmark = run_benchmark(
        "board/vivado_zynq7000_xc7z020_pynq/release",
        warmup=5,
        repetitions=30,
        kat_count=529,
    )

print(benchmark_log.getvalue())
assert benchmark["board_validation"] == "PASS"
(run_dir / "pynq_warm_benchmark.json").write_text(
    json.dumps(benchmark, indent=2) + "\n", encoding="utf-8"
)
(run_dir / "pynq_warm_benchmark.log").write_text(
    benchmark_log.getvalue(), encoding="utf-8"
)
print("benchmark evidence saved to", run_dir)

sw encrypt median=23342.0, decrypt median=24005.0
hw encrypt median=1439.0, decrypt median=1388.0
ise encrypt median=3995.0, decrypt median=4385.0

benchmark evidence saved to /home/xilinx/jupyter_notebooks/amd_fpga_2026/data/pynq/notebook_20261031_112052


## 3. 结果摘要

cycles 用于 SW/HW/ISE 对比；wall time 只用于观察 PS 端到端开销，不能替代 `mcycle`。

In [4]:
print("mode   op       median_cycles  p95_cycles  median_wall_ms")
for mode_name in ("sw", "hw", "ise"):
    for operation in ("encrypt", "decrypt"):
        item = benchmark["modes"][mode_name][operation]
        print("{:<6} {:<8} {:>13} {:>11} {:>15.3f}".format(
            mode_name,
            operation,
            item["summary"]["median"],
            item["summary"]["p95"],
            item["wall_ns_summary"]["median"] / 1_000_000,
        ))
print("speedup median vs SW:", benchmark["speedup_median_vs_sw"])

mode   op       median_cycles  p95_cycles  median_wall_ms
sw     encrypt        23342.0       23342           5.152
sw     decrypt        24005.0       24005           4.899
hw     encrypt         1439.0        1439           4.422
hw     decrypt         1388.0        1388           4.193
ise    encrypt         3995.0        3995           5.097
ise    decrypt         4385.0        4385           4.893
speedup median vs SW: {'hw': {'encrypt': 16.221, 'decrypt': 17.295}, 'ise': {'encrypt': 5.843, 'decrypt': 5.474}}


本次 `RUN_FULL_KAT = True`，以上结果覆盖三模式容量内全部 289 组 KAT。
若只需快速回归，可将 `RUN_FULL_KAT` 改为 `False`，此时每模式运行 6 个代表用例。
超过 16-byte 的 800 组仍属于 `NOT_RUN`，不能声明硬件全量通过。
